<head>
    <style>
        .md-typeset h2 {
            margin:0;
            }
        .md-typeset h3 {
            margin:0;
            }
        .jupyter-wrapper table.dataframe tr, .jupyter-wrapper table.dataframe th, .jupyter-wrapper table.dataframe td {
            text-align:left;
            }
        .jupyter-wrapper table.dataframe {
            table-layout: auto;
            }
    </style>        
</head>

# Data Manufacturing 

## Introduction 

In this post, we will write up a `class` to process all the data we need from 
the sec. This `class` could be taken as a data driven manufacture that provides
all functions we need. Here is the design for this `class`:

* it should download both `firm facts` and `firm info` based on `CIK` number
* it should clean the data for us automatically
* it should find out the common account items 
* it should have functions for calculating key ratios and plot results

In [2]:
# import packages
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import json
import random
sns.set_theme(context='notebook', style='darkgrid')

# requests headers
headers = """
authority: data.sec.gov
method: GET
scheme: https
accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9
accept-encoding: gzip, deflate, br
accept-language: en-US,en;q=0.9
cache-control: max-age=0
sec-fetch-dest: document
sec-fetch-mode: navigate
sec-fetch-site: none
sec-fetch-user: ?1
upgrade-insecure-requests: 1
user-agent: Mozilla/5.0 (iPhone; CPU iPhone OS 13_2_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/13.0.3 Mobile/15E148 Safari/604.1 Edg/103.0.5060.114
"""

headers = headers.strip().split('\n')
# save it as dict
HEADERS = {x.split(':')[0].strip():
           ("".join(x.split(':')[1:])).strip().replace('//',
                                                       "://") for x in headers}

In [70]:
class SecEdgar:
    """
    A class for processing data from SEC based on CIK number 
    """
    
    def __init__(self, cik):
        self.cik = cik
        self.facts = None
        self.info = None
        self.recent_filings = None
        self.all_filings = None
        self.common_accounts = None
        self.common_start_year = None
        self.common_end_year = None
        
    def download(self):
        '''
        it will download company facts and submission information 
        '''
        facts_url = 'https://data.sec.gov/api/xbrl/companyfacts/CIK'
        facts_url = facts_url + self.cik + '.json'
        facts_r = requests.get(facts_url, headers=HEADERS)
        self.facts = facts_r.json()
        
        info_url = 'https://data.sec.gov/submissions/CIK'
        info_url = info_url + self.cik + '.json'
        info_r = requests.get(info_url, headers=HEADERS)
        self.info = info_r.json()
        
    # private method 
    def __facts_to_df(self, gaap_tag):
        '''
        Input
            firm_json: json file downloaded from sec
            gaap_tag: account tag
        Output
            dataframe for given account tag
        '''
        firm_account = self.facts['facts']['us-gaap'][gaap_tag]
        tag = list(firm_account['units'].keys())[0]
        firm_df = pd.DataFrame(firm_account['units'].get(tag))
        return firm_df
    
    def check_account(self, gaap_tag):
        return self.__facts_to_df(gaap_tag)
        
    def __collect_filings(self, start_year, end_year):
        '''
        A method to find accounts of filings
        Input
            self.facts
            start_year(int)
            end_year(int)
        Output
            a dict with keys=year and values=accounts available in that year
        '''
        # initialize the dictionary
        firm_10k_dict = {}
        for year in range(start_year, end_year):
            firm_10k_dict[str(year)] = []
            for accounts in self.facts['facts']['us-gaap'].keys():
                try:
                    temp_df = self.__facts_to_df(accounts)
                    temp_df_10k = temp_df[temp_df['form'] == '10-K']
                    temp_df_10k = temp_df_10k.drop_duplicates(subset='end')
                    condition = any(temp_df_10k['end'].str.contains(str(year)))
                    if condition:
                        firm_10k_dict[str(year)].append(accounts)
                except KeyError:
                    print(KeyError, accounts)
                    
        return firm_10k_dict
        
    def collect_recent_filings(self):
        end_year = self.info['filings']['recent']['filingDate'][0][:4]
        end_year = int(end_year)
        start_year = self.info['filings']['recent']['filingDate'][-1][:4]
        start_year = int(start_year)
        
        self.recent_filings = self.__collect_filings(start_year, end_year)
        
    def collect_all_filings(self):
        end_year = self.info['filings']['recent']['filingDate'][0][:4]
        end_year = int(end_year)
        start_year = self.info['filings']['files'][0]['filingFrom'][:4]
        start_year = int(start_year)
        
        self.all_filings = self.__collect_filings(start_year, end_year)
        
    def find_recent_common(self,num_of_accounts):
        '''
        Input
            self.recent_filings
            num_of_accounts: number of accounts at least
        Output
            a set of common accounts for all recent years
        '''
        temp_dict = {}
        for key, value in self.recent_filings.items():
            if len(value) >= num_of_accounts:
                temp_dict[key] = value

        common_accounts = set(list(temp_dict.values())[0])
        for accounts in temp_dict.values():
            common_accounts = common_accounts.intersection(set(accounts))
        
        self.common_accounts = common_accounts
        self.common_start_year = list(temp_dict.keys())[0]
        self.common_end_year = list(temp_dict.keys())[-1]

In [71]:
# test cik = 0000320193
aapl_cik = '0000320193'
aapl = SecEdgar(aapl_cik)

In [72]:
aapl.download()

In [77]:
aapl.collect_recent_filings()

In [54]:
for key, value in aapl.recent_filings.items():
    print(key, len(value))

2012 235
2013 244
2014 227
2015 226
2016 220
2017 237
2018 225
2019 203
2020 217
2021 205
